# Business Entity Resolution: Amazon ML Challenge 2026

End-to-end pipeline that writes `output/matching_results.tsv` (upload this to the portal) and
`output/candidate_pairs.tsv` (for the final zip) into `/kaggle/working/output/`.

**Setup**
1. Add the competition data as a Kaggle Dataset. The notebook finds `train_source1.tsv` anywhere under `/kaggle/input`, so the folder name does not matter.
   Optionally include `utils/validate_submission.py` in the dataset; it will be run at the end.
2. Settings: **Accelerator = GPU T4 x2 (or P100)**, **Internet = On** (needed once for `pip install`).
3. Run all. Expected runtime is about 2.5–4 h, mostly CPU blocking on Kaggle's 4 cores. Peak RAM is about 12–16 GB, within the ~29 GB free tier.
   Each stage caches to `/kaggle/working/cache`, so an interrupted run resumes where it stopped.

**Method**
1. Normalize names and addresses, inverting the synthetic noise: legal suffixes, DBA/formerly aliases, domains, honorifics, accents, street/state abbreviations.
2. Romanize native scripts with anyascii, then translate them through a lexicon mined from training matches.
3. Blocking: per-country TF-IDF over name and address tokens. Each S2/S3 record retrieves its top-10 S1, because every S2/S3 record matches at most one S1.
4. XGBoost (GPU) pair classifier on about 45 string, token-overlap and competition features. Country is not a feature, so the model transfers to France.
5. Assignment decoding: each S2/S3 record goes to its argmax S1 only if p > τ. τ is tuned for macro F0.5 on held-out S1 entities, at full S1 density.

In [1]:
!pip install -q rapidfuzz anyascii sparse_dot_topn 2>&1 | tail -2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.6/309.6 kB 17.2 MB/s eta 0:00:00


In [2]:
import gc, glob, json, os, pickle, re, subprocess, sys, time
from collections import Counter, defaultdict
from multiprocessing import Pool

import numpy as np
import pandas as pd
import xgboost as xgb
from anyascii import anyascii
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from rapidfuzz.process import cpdist
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn

DEBUG = int(os.environ.get("ER_DEBUG", "0"))       # 1 = tiny subsample, for smoke tests only
ON_KAGGLE = os.path.isdir("/kaggle/input")


def find_data_dir():
    if os.environ.get("ER_DATA"):
        return os.environ["ER_DATA"]
    hits = glob.glob("/kaggle/input/**/train_source1.tsv", recursive=True)
    assert hits, "train_source1.tsv not found under /kaggle/input: add the dataset to the notebook"
    return os.path.dirname(os.path.dirname(hits[0]))   # .../<root>/train/train_source1.tsv -> <root>


DATA = find_data_dir()
WORK = "/kaggle/working" if ON_KAGGLE else os.path.abspath("kaggle_work")
if DEBUG:
    WORK = os.path.join(WORK, "debug")
CACHE, OUT = os.path.join(WORK, "cache"), os.path.join(WORK, "output")
os.makedirs(CACHE, exist_ok=True); os.makedirs(OUT, exist_ok=True)
N_JOBS = os.cpu_count() or 2
try:
    GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except FileNotFoundError:
    GPU = False
SEED = 0
CFG = dict(
    k=10,              # candidates retrieved per S2/S3 record
    max_df=0.02,       # blocking ignores tokens present in >2% of a country's S1 (speed)
    rel=0.4,           # drop candidates scoring < 40% of the record's best candidate
    frac=0.06,         # training: S1 subset E whose matches are all included
    other_rate=0.12,   # training: fraction of all other S2/S3 records included (distractors)
    rounds=1500,
)
T0 = time.time()


def log(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)


def ram():
    try:
        with open("/proc/self/status") as f:
            rss = [l for l in f if l.startswith("VmRSS")][0].split()[1]
        return f"{int(rss) / 1e6:.1f} GB"
    except Exception:
        return "?"


log(f"data={DATA} work={WORK} cpus={N_JOBS} gpu={GPU} debug={DEBUG}")

[   0.0 min] data=/kaggle/input/datasets/gouravanirudh/amazon-ml-challenge/student_resource/dataset work=/kaggle/working cpus=4 gpu=True debug=0


## 1. Normalization

In [3]:
LEGAL = {
    "pvt": "private", "private": "private", "ltd": "limited", "limited": "limited",
    "llc": "llc", "llp": "llp", "lp": "lp", "plc": "plc", "inc": "inc",
    "incorporated": "inc", "corp": "corp", "corporation": "corp", "co": "co",
    "company": "co", "sarl": "sarl", "sas": "sas", "sasu": "sasu", "eurl": "eurl",
    "sa": "sa", "sci": "sci", "snc": "snc", "gmbh": "gmbh", "pllc": "pllc",
}
STOP = {"the", "and", "mr", "mrs", "ms", "dr", "sri", "shri", "smt", "m/s", "of",
        "et", "le", "la", "les", "de", "du", "des"}
ALIAS_RE = re.compile(r"\b(?:dba|d/b/a|formerly|aka|f/k/a|fka)\b[:\s]*", re.I)
DOMAIN_RE = re.compile(r"^(?:www\.)?([a-z0-9\-]+)\.(?:com|in|net|org|co|fr|biz|info|us|co\.in)$")
NON_ALNUM = re.compile(r"[^a-z0-9]+")
NUM_RE = re.compile(r"\d+")
NATIVE_RE = re.compile("[ऀ-෿]")        # Devanagari ... Sinhala blocks
ADDR_ABBR = {
    "street": "st", "saint": "st", "road": "rd", "avenue": "ave", "av": "ave", "drive": "dr",
    "circle": "cir", "court": "ct", "boulevard": "blvd", "bd": "blvd", "lane": "ln",
    "place": "pl", "highway": "hwy", "parkway": "pkwy", "terrace": "ter", "trail": "trl",
    "square": "sq", "north": "n", "south": "s", "east": "e", "west": "w",
    "apartment": "apt", "suite": "ste", "floor": "fl", "number": "no", "building": "bldg",
    "chemin": "ch", "impasse": "imp", "allee": "all", "route": "rte",
    "sector": "sec", "opposite": "opp", "near": "nr", "nagar": "ngr",
}
US_STATES = {
    "alabama": "al", "alaska": "ak", "arizona": "az", "arkansas": "ar", "california": "ca",
    "colorado": "co", "connecticut": "ct", "delaware": "de", "florida": "fl", "georgia": "ga",
    "hawaii": "hi", "idaho": "id", "illinois": "il", "indiana": "in", "iowa": "ia",
    "kansas": "ks", "kentucky": "ky", "louisiana": "la", "maine": "me", "maryland": "md",
    "massachusetts": "ma", "michigan": "mi", "minnesota": "mn", "mississippi": "ms",
    "missouri": "mo", "montana": "mt", "nebraska": "ne", "nevada": "nv",
    "new hampshire": "nh", "new jersey": "nj", "new mexico": "nm", "new york": "ny",
    "north carolina": "nc", "north dakota": "nd", "ohio": "oh", "oklahoma": "ok",
    "oregon": "or", "pennsylvania": "pa", "rhode island": "ri", "south carolina": "sc",
    "south dakota": "sd", "tennessee": "tn", "texas": "tx", "utah": "ut", "vermont": "vt",
    "virginia": "va", "washington": "wa", "west virginia": "wv", "wisconsin": "wi",
    "wyoming": "wy", "district of columbia": "dc",
}
STATE_RE = re.compile(r"\b(" + "|".join(sorted(US_STATES, key=len, reverse=True)) + r")\b")


def norm_name(raw):
    # returns (core, full, legal, flags); flags: 1 native script, 2 alias, 4 domain
    s = anyascii(raw).lower() if raw else ""
    flags = 1 if NATIVE_RE.search(raw or "") else 0
    m = ALIAS_RE.search(s)
    if m:
        flags |= 2
        s = s[m.end():] or s[: m.start()]
    s = s.strip()
    d = DOMAIN_RE.match(s)
    if d:
        flags |= 4
        s = d.group(1)
    else:
        s = s.replace("&", " and ").replace(".", "")
    toks = NON_ALNUM.sub(" ", s).split()
    core = [t for t in toks if t not in LEGAL and t not in STOP] or toks
    legal = sorted({LEGAL[t] for t in toks if t in LEGAL})
    return " ".join(core), " ".join(toks), " ".join(legal), flags


def norm_addr(raw):
    s = (anyascii(raw).lower() if raw else "").replace(".", " ")
    s = STATE_RE.sub(lambda m: US_STATES[m.group(1)], s)
    toks = [ADDR_ABBR.get(t, t) for t in NON_ALNUM.sub(" ", s).split()]
    toks = [t for t in toks if t not in ("null", "none", "na")]
    return " ".join(toks), " ".join(sorted(set(NUM_RE.findall(" ".join(toks)))))


NORM_COLS = ["core", "full", "legal", "flags", "addr", "nums"]


def _norm_chunk(args):
    names, addrs = args
    rows = [norm_name(n) + norm_addr(a) for n, a in zip(names, addrs)]
    df = pd.DataFrame(rows, columns=NORM_COLS)
    df["flags"] = df["flags"].astype(np.int8)
    return df.astype({c: "string[pyarrow]" for c in NORM_COLS if c != "flags"})


def read_tsv(path):
    return pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, quoting=3)


def load_source(split, src):
    # normalized source file with ids + country only (raw text dropped to save RAM); cached
    cp = os.path.join(CACHE, f"{split}_s{src}.parquet")
    if os.path.exists(cp):
        return pd.read_parquet(cp, dtype_backend="pyarrow")
    df = read_tsv(os.path.join(DATA, split, f"{split}_source{src}.tsv"))
    names, addrs = df.business_name.tolist(), df.business_address.tolist()
    step = 100_000
    chunks = [(names[i:i + step], addrs[i:i + step]) for i in range(0, len(df), step)]
    del names, addrs
    with Pool(N_JOBS) as pool:
        norm = pd.concat(pool.imap(_norm_chunk, chunks), ignore_index=True)
    out = pd.concat([df[["entity_id", "country"]].astype("string[pyarrow]").reset_index(drop=True), norm], axis=1)
    del df, chunks, norm; gc.collect()
    out.to_parquet(cp, index=False)
    return out


def load_split(split):
    s1 = load_source(split, 1)
    q = pd.concat([load_source(split, 2), load_source(split, 3)], ignore_index=True)
    q["is_s3"] = q.entity_id.str.startswith("S3-").astype(np.int8)
    return s1, q


def load_truth():
    # Series: S2/S3 id -> its S1 id (each S2/S3 matches at most one S1)
    gt = read_tsv(os.path.join(DATA, "train", "train_ground_truth.tsv"))
    s, m = [], []
    for a, b in zip(gt.source1_entity_id, gt.matched_entity_ids):
        for x in (b.split(",") if b else []):
            s.append(a); m.append(x)
    return pd.Series(s, index=m)


for sp in ("train", "test"):
    for src in (1, 2, 3):
        n = len(load_source(sp, src)); gc.collect()
        log(f"normalized {sp} source{src}: {n} rows, RAM {ram()}")

[   0.5 min] normalized train source1: 2206821 rows, RAM 0.8 GB
[   1.7 min] normalized train source2: 5034616 rows, RAM 1.3 GB
[   2.9 min] normalized train source3: 5285603 rows, RAM 1.4 GB
[   3.3 min] normalized test source1: 1732544 rows, RAM 1.5 GB
[   4.5 min] normalized test source2: 4887273 rows, RAM 2.0 GB
[   5.7 min] normalized test source3: 5082316 rows, RAM 2.0 GB


## 2. Transliteration lexicon (mined from training matches)

In [4]:
def mine_lexicon(s1, q, truth, min_count=3, min_share=0.5):
    # romanized native-script token -> dominant co-occurring Source-1 token ("knstrksn" -> "construction")
    native = q[(q["flags"].astype(int) & 1) == 1]
    s1_full = pd.Series(s1["full"].astype(str).to_numpy(), index=s1["entity_id"].astype(str))
    tgt = native["entity_id"].astype(str).map(truth)
    ok = tgt.notna().to_numpy()
    src_names = native["full"].astype(str).to_numpy()[ok]
    tgt_names = s1_full.reindex(tgt[ok]).fillna("").to_numpy()
    co, cnt = defaultdict(Counter), Counter()
    for a, b in zip(src_names, tgt_names):
        bt = set(b.split())
        for t in set(a.split()):
            cnt[t] += 1
            co[t].update(bt)
    lex = {}
    for t, c in cnt.items():
        if c >= min_count:
            best, n = co[t].most_common(1)[0]
            if n / c >= min_share and best != t:
                lex[t] = best
    return lex


def apply_lexicon(df, lex):
    mask = ((df["flags"].astype(int) & 1) == 1).to_numpy()
    cores, fulls, legals = [], [], []
    for f in df.loc[mask, "full"].astype(str).tolist():
        toks = " ".join(lex.get(t, t) for t in f.split()).split()
        cores.append(" ".join([t for t in toks if t not in LEGAL and t not in STOP] or toks))
        fulls.append(" ".join(toks))
        legals.append(" ".join(sorted({LEGAL[t] for t in toks if t in LEGAL})))
    for col, vals in (("core", cores), ("full", fulls), ("legal", legals)):
        df.loc[mask, col] = pd.array(vals, dtype=df[col].dtype)
    return df


LEX_PATH = os.path.join(CACHE, "lexicon.pkl")
if not os.path.exists(LEX_PATH):
    _s1, _q = load_split("train")
    LEX = mine_lexicon(_s1, _q, load_truth())
    pickle.dump(LEX, open(LEX_PATH, "wb"))
    del _s1, _q; gc.collect()
LEX = pickle.load(open(LEX_PATH, "rb"))
log(f"lexicon {len(LEX)} tokens, e.g. " + ", ".join(f"{k}->{v}" for k, v in list(LEX.items())[:8]))

[   6.1 min] lexicon 571 tokens, e.g. modrn->modern, riyl->real, phud->food, kulopl->global, piraivet->limited, limitet->limited, picins->business, prodkts->products


## 3. Blocking: each S2/S3 record retrieves its top-k S1 within its country

In [5]:
def _docs(df):
    name = df["core"].fillna("").astype(str).str.replace(r"(\S+)", r"n_\1", regex=True)
    addr = df["addr"].fillna("").astype(str).str.replace(r"(\S+)", r"a_\1", regex=True)
    return name + " " + addr


def block(s1, q, k=CFG["k"], max_df=CFG["max_df"], rel=CFG["rel"], chunk=400_000):
    # candidate pairs (qi, si, score, rank), sorted by qi then rank; identical docs are retrieved once
    out = []
    q_ctry, s_ctry = q["country"].astype(str).to_numpy(), s1["country"].astype(str).to_numpy()
    for country in pd.unique(q_ctry):                    # open set of countries, taken from the data
        s_idx, q_idx = np.flatnonzero(s_ctry == country), np.flatnonzero(q_ctry == country)
        if len(s_idx) == 0 or len(q_idx) == 0:
            continue
        vec = TfidfVectorizer(token_pattern=r"\S+", lowercase=False, sublinear_tf=True,
                              max_df=max_df, dtype=np.float32)
        ST = vec.fit_transform(_docs(s1.iloc[s_idx]).tolist()).T.tocsr()
        codes, uniq = pd.factorize(_docs(q.iloc[q_idx]))
        log(f"  block {country}: {len(s_idx)} S1, {len(q_idx)} queries ({len(uniq)} unique docs)")
        r_row, r_col, r_val = [], [], []
        for a in range(0, len(uniq), chunk):
            Q = vec.transform(list(uniq[a:a + chunk]))
            C = sp_matmul_topn(Q, ST, top_n=k, sort=True, threshold=0.05, n_threads=N_JOBS).tocoo()
            r_row.append(C.row.astype(np.int64) + a); r_col.append(C.col.astype(np.int32)); r_val.append(C.data)
        rows, cols, vals = np.concatenate(r_row), np.concatenate(r_col), np.concatenate(r_val)
        o = np.lexsort((-vals, rows)); rows, cols, vals = rows[o], cols[o], vals[o]
        # per unique doc: [start, start+size) slice, pruned relative to its best score
        starts = np.searchsorted(rows, np.arange(len(uniq)))
        sizes = np.diff(np.r_[starts, len(rows)])
        rank = np.arange(len(rows)) - np.repeat(starts, sizes)
        keep = vals >= rel * np.repeat(vals[np.minimum(starts, len(vals) - 1)], sizes)
        cols, vals, rank, rows = cols[keep], vals[keep], rank[keep], rows[keep]
        starts = np.searchsorted(rows, np.arange(len(uniq)))
        sizes = np.diff(np.r_[starts, len(rows)])
        # expand unique docs back to every query that has that doc
        qs = sizes[codes]
        pos = np.repeat(starts[codes], qs) + (np.arange(qs.sum()) - np.repeat(np.cumsum(qs) - qs, qs))
        out.append(pd.DataFrame({"qi": np.repeat(q_idx, qs).astype(np.int32),
                                 "si": s_idx[cols[pos]].astype(np.int32),
                                 "score": vals[pos].astype(np.float32),
                                 "rank": rank[pos].astype(np.int8)}))
        del ST, vec, rows, cols, vals, rank, pos; gc.collect()
        log(f"  block {country}: done, {len(out[-1])} pairs, RAM {ram()}")
    P = pd.concat(out, ignore_index=True)
    P = P.sort_values(["qi", "rank"], kind="stable").reset_index(drop=True)
    return add_context(P)


def add_context(P):
    # per-query context: best score, gap to runner-up, number of candidates
    qi, sc = P["qi"].to_numpy(), P["score"].to_numpy()
    starts = np.r_[0, np.flatnonzero(np.diff(qi)) + 1]
    sizes = np.diff(np.r_[starts, len(qi)])
    best = np.repeat(sc[starts], sizes)
    second = np.repeat(np.where(sizes > 1, sc[np.minimum(starts + 1, len(sc) - 1)], 0.0), sizes)
    P["q_best_ratio"] = (sc / np.maximum(best, 1e-6)).astype(np.float32)
    P["q_gap_best"] = (best - sc).astype(np.float32)
    P["q_gap_next"] = np.where(P["rank"].to_numpy() == 0, sc - second, sc - best).astype(np.float32)
    P["q_ncand"] = np.repeat(sizes, sizes).astype(np.int16)
    return P

## 4. Pair features

In [6]:
CTX = ["score", "rank", "q_best_ratio", "q_gap_best", "q_gap_next", "q_ncand"]


class Featurizer:
    # country is deliberately not a feature, so the model transfers to unseen countries (France)
    def __init__(self, s1, q):
        cols = ("core", "full", "legal", "addr", "nums")
        self.s = {c: s1[c].fillna("").astype(str).to_numpy(dtype=object) for c in cols}
        self.q = {c: q[c].fillna("").astype(str).to_numpy(dtype=object) for c in cols}
        self.q_flags = q["flags"].astype(np.int8).to_numpy()
        self.q_s3 = q["is_s3"].astype(np.int8).to_numpy()
        key = s1["country"].astype(str) + "|" + s1["core"].astype(str)
        self.s_name_freq = key.map(key.value_counts()).astype(np.int32).to_numpy()
        self.mats = {}
        for c in ("core", "addr", "nums"):     # binary token matrices -> vectorized overlap counts
            cv = CountVectorizer(token_pattern=r"\S+", lowercase=False, binary=True, dtype=np.float32)
            S = cv.fit_transform(self.s[c]).tocsr()
            Q = cv.transform(self.q[c]).tocsr()
            ls = np.array([x.count(" ") + 1 if x else 0 for x in self.s[c]], dtype=np.float32)
            lq = np.array([x.count(" ") + 1 if x else 0 for x in self.q[c]], dtype=np.float32)
            self.mats[c] = (S, Q, ls, lq)

    def _overlap(self, c, qi, si):
        S, Q, ls, lq = self.mats[c]
        inter = np.asarray(Q[qi].multiply(S[si]).sum(1)).ravel()
        union = lq[qi] + ls[si] - inter
        return inter, np.where(union > 0, inter / np.maximum(union, 1), 0), lq[qi], ls[si]

    def transform(self, P):
        qi, si = P["qi"].to_numpy(), P["si"].to_numpy()
        F = {c: P[c].to_numpy() for c in CTX}
        kw = dict(workers=N_JOBS, dtype=np.float32)
        qc, sc = self.q["core"][qi], self.s["core"][si]
        F["n_ratio"] = cpdist(qc, sc, scorer=fuzz.ratio, **kw)
        F["n_tsort"] = cpdist(qc, sc, scorer=fuzz.token_sort_ratio, **kw)
        F["n_tset"] = cpdist(qc, sc, scorer=fuzz.token_set_ratio, **kw)
        F["n_partial"] = cpdist(qc, sc, scorer=fuzz.partial_ratio, **kw)
        F["n_jw"] = cpdist(qc, sc, scorer=JaroWinkler.normalized_similarity, **kw)
        qsq = np.array([x.replace(" ", "") for x in qc], dtype=object)
        ssq = np.array([x.replace(" ", "") for x in sc], dtype=object)
        F["n_squash_ratio"] = cpdist(qsq, ssq, scorer=fuzz.ratio, **kw)
        F["n_squash_partial"] = cpdist(qsq, ssq, scorer=fuzz.partial_ratio, **kw)
        F["n_full_tsort"] = cpdist(self.q["full"][qi], self.s["full"][si], scorer=fuzz.token_sort_ratio, **kw)
        inter, jac, lq, ls = self._overlap("core", qi, si)
        F.update(n_inter=inter, n_jac=jac, n_lq=lq, n_ls=ls, n_missing_s=ls - inter, n_missing_q=lq - inter)
        ql, sl = self.q["legal"][qi], self.s["legal"][si]
        both = (ql != "") & (sl != "")
        F["legal_both"] = both.astype(np.int8)
        F["legal_eq"] = (both & (ql == sl)).astype(np.int8)
        F["legal_sim"] = cpdist(ql, sl, scorer=fuzz.token_set_ratio, **kw) * both
        qa, sa = self.q["addr"][qi], self.s["addr"][si]
        F["a_ratio"] = cpdist(qa, sa, scorer=fuzz.ratio, **kw)
        F["a_tset"] = cpdist(qa, sa, scorer=fuzz.token_set_ratio, **kw)
        F["a_tsort"] = cpdist(qa, sa, scorer=fuzz.token_sort_ratio, **kw)
        F["a_partial"] = cpdist(qa, sa, scorer=fuzz.partial_ratio, **kw)
        inter, jac, lq, ls = self._overlap("addr", qi, si)
        F.update(a_inter=inter, a_jac=jac, a_lq=lq, a_ls=ls)
        inter, jac, lq, ls = self._overlap("nums", qi, si)
        F.update(num_inter=inter, num_jac=jac, num_lq=lq, num_ls=ls)
        F["num_tset"] = cpdist(self.q["nums"][qi], self.s["nums"][si], scorer=fuzz.token_set_ratio, **kw)
        flags = self.q_flags[qi]
        F["q_native"], F["q_alias"], F["q_domain"] = flags & 1, (flags >> 1) & 1, (flags >> 2) & 1
        F["q_is_s3"] = self.q_s3[qi]
        F["s_name_freq"] = self.s_name_freq[si]
        # competition: this candidate vs. the query's best alternative (chunks never split a query)
        for c in ("n_tsort", "n_squash_ratio", "a_tset", "num_jac"):
            v = pd.Series(np.asarray(F[c], dtype=np.float32))
            F[c + "_gap_qmax"] = (v.groupby(qi).transform("max").to_numpy() - v.to_numpy()).astype(np.float32)
        return pd.DataFrame(F)


def chunk_bounds(qi, size):
    # ~size-row chunks that never split one query's candidates
    cuts = [0]
    while cuts[-1] < len(qi):
        c = min(cuts[-1] + size, len(qi))
        while 0 < c < len(qi) and qi[c] == qi[c - 1]:
            c += 1
        cuts.append(c)
    return list(zip(cuts[:-1], cuts[1:]))


def featurize(feat, P, size=1_000_000):
    X, names = None, None
    for a, b in chunk_bounds(P["qi"].to_numpy(), size):
        F = feat.transform(P.iloc[a:b])
        if X is None:
            names = list(F.columns)
            X = np.empty((len(P), len(names)), dtype=np.float32)
        X[a:b] = F.to_numpy(dtype=np.float32)
    return X, names


def decode(qi, prob, t):
    # assignment constraint: each S2/S3 record goes to its argmax S1 iff prob > t
    order = np.lexsort((-prob, qi))
    first = np.r_[True, qi[order][1:] != qi[order][:-1]]
    best = order[first]
    return best[prob[best] > t]

## 5. Training at full S1 density

A query sample is blocked against **all** training S1, so competition between similar S1 records is realistic.
The sample is every S2/S3 record matched to a random S1 subset E, plus a fraction of all other records.
Macro F0.5 is measured on held-out S1 entities V ⊂ E, with sampled false positives up-weighted to population scale.

In [7]:
MODEL_PATH, REPORT_PATH = os.path.join(CACHE, "xgb_model.json"), os.path.join(CACHE, "train_report.json")
XGB_PARAMS = dict(objective="binary:logistic", eval_metric="logloss", tree_method="hist",
                  device="cuda" if GPU else "cpu", max_depth=10, learning_rate=0.08, subsample=0.8,
                  colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0, max_bin=256, seed=SEED,
                  nthread=N_JOBS)

if not os.path.exists(REPORT_PATH):
    rng = np.random.default_rng(SEED)
    s1, q = load_split("train")
    apply_lexicon(q, LEX)
    truth = load_truth()
    frac, other_rate = CFG["frac"], CFG["other_rate"]
    if DEBUG:                               # smoke test: tiny S1 world
        keep_s1 = rng.random(len(s1)) < 0.05
        ids = set(s1.entity_id.astype(str)[keep_s1])
        tq = q.entity_id.astype(str).map(truth)
        q = q[(tq.isin(ids) | (tq.isna() & (rng.random(len(q)) < 0.05))).to_numpy()].reset_index(drop=True)
        s1 = s1[keep_s1].reset_index(drop=True)
        frac, other_rate = 0.3, 0.5
    s1id = s1.entity_id.astype(str).to_numpy()
    in_e = rng.random(len(s1)) < frac
    in_v = in_e & (rng.random(len(s1)) < 0.25)
    pos = pd.Series(np.arange(len(s1)), index=s1id)
    t_pos = q.entity_id.astype(str).map(truth).map(pos).fillna(-1).astype(np.int64).to_numpy()
    q_e = (t_pos >= 0) & in_e[np.maximum(t_pos, 0)]
    keep = q_e | (~q_e & (rng.random(len(q)) < other_rate))
    q, t_pos, q_e = q[keep].reset_index(drop=True), t_pos[keep], q_e[keep]
    q_val = np.where(q_e, in_v[np.maximum(t_pos, 0)], rng.random(len(q)) < 0.25)
    log(f"train sample: {len(q)} queries ({q_e.sum()} matched to E), E={in_e.sum()} V={in_v.sum()} S1")

    P = block(s1, q)
    pqi, psi = P.qi.to_numpy(), P.si.to_numpy()
    y = (t_pos[pqi] == psi).astype(np.float32)
    has = t_pos >= 0
    found = np.zeros(len(q), bool); found[pqi[y == 1]] = True
    log(f"pairs {len(P)} ({len(P) / len(q):.2f}/query), blocking recall {found[has].mean():.4f}")

    order = np.argsort(q_val[pqi], kind="stable")          # train rows first, val rows last
    P, y = P.iloc[order].reset_index(drop=True), y[order]
    pqi, psi = P.qi.to_numpy(), P.si.to_numpy()
    n_tr = int((~q_val[pqi]).sum())
    feat = Featurizer(s1, q)
    X, FEATURES = featurize(feat, P)
    del feat; gc.collect()
    w = np.where(q_e[pqi], 1.0 / frac, 1.0 / other_rate).astype(np.float32)
    w /= w.mean()
    log(f"features {X.shape}, train rows {n_tr}, RAM {ram()}")

    dtr = xgb.QuantileDMatrix(X[:n_tr], y[:n_tr], weight=w[:n_tr], feature_names=FEATURES)
    dva = xgb.QuantileDMatrix(X[n_tr:], y[n_tr:], weight=w[n_tr:], ref=dtr, feature_names=FEATURES)
    bst = xgb.train(XGB_PARAMS, dtr, num_boost_round=CFG["rounds"] if not DEBUG else 60,
                    evals=[(dva, "val")], early_stopping_rounds=50, verbose_eval=100)
    prob = bst.predict(dva, iteration_range=(0, bst.best_iteration + 1)).astype(np.float32)
    del X, dtr, dva; gc.collect()

    vqi, vsi = pqi[n_tr:], psi[n_tr:]
    v_idx = np.flatnonzero(in_v)
    n_true = np.bincount(t_pos[has & q_e & q_val], minlength=len(s1))[v_idx]
    w_fp = np.where(q_e, 1.0, 1.0 / (other_rate * 0.25))   # sampled distractors -> population scale

    def val_f05(t):
        sel = decode(vqi, prob, t)
        s_sel, q_sel = vsi[sel], vqi[sel]
        m = in_v[s_sel]; s_sel, q_sel = s_sel[m], q_sel[m]
        ok = (t_pos[q_sel] == s_sel).astype(float)
        tp = np.bincount(s_sel, weights=ok, minlength=len(s1))[v_idx]
        npred = np.bincount(s_sel, weights=ok + (1 - ok) * w_fp[q_sel], minlength=len(s1))[v_idx]
        return np.where(n_true == 0, (npred == 0).astype(float),
                        1.25 * tp / np.maximum(npred + 0.25 * n_true, 1e-9))

    grid = np.round(np.arange(0.30, 0.985, 0.01), 3)
    scores = [val_f05(t).mean() for t in grid]
    best_t = float(grid[int(np.argmax(scores))])
    fs = val_f05(best_t)
    ctry = s1.country.astype(str).to_numpy()[v_idx]
    report = {"threshold": best_t, "val_macro_f05": float(fs.mean()),
              "val_f05_singletons": float(fs[n_true == 0].mean()) if (n_true == 0).any() else None,
              "val_f05_non_singletons": float(fs[n_true > 0].mean()),
              "val_f05_by_country": {c: float(fs[ctry == c].mean()) for c in np.unique(ctry)},
              "blocking_recall": float(found[has].mean()), "best_iteration": int(bst.best_iteration),
              "features": FEATURES}
    bst.save_model(MODEL_PATH)
    json.dump(report, open(REPORT_PATH, "w"), indent=1)
    imp = pd.Series(bst.get_score(importance_type="gain")).sort_values(ascending=False)
    print((imp / imp.sum()).head(15).round(4).to_string())
    print("F0.5 vs threshold:", {float(t): round(s, 4) for t, s in zip(grid[::5], scores[::5])})
    del s1, q, P, prob, bst; gc.collect()

REPORT = json.load(open(REPORT_PATH))
print(json.dumps({k: v for k, v in REPORT.items() if k != "features"}, indent=1))

[   6.7 min] train sample: 1641459 queries (457393 matched to E), E=132209 V=32755 S1
[   7.2 min]   block India: 883188 S1, 656784 queries (643559 unique docs)
[   9.8 min]   block India: done, 4278545 pairs, RAM 4.8 GB
[  10.2 min]   block US: 1323633 S1, 984675 queries (944256 unique docs)
[  13.7 min]   block US: done, 6117594 pairs, RAM 5.1 GB
[  13.7 min] pairs 10396139 (6.33/query), blocking recall 0.9722
[  16.8 min] features (10396139, 45), train rows 7802386, RAM 6.4 GB
[0]	val-logloss:0.31736
[100]	val-logloss:0.01474
[200]	val-logloss:0.01322
[300]	val-logloss:0.01273
[400]	val-logloss:0.01254
[500]	val-logloss:0.01244
[600]	val-logloss:0.01238
[700]	val-logloss:0.01235
[799]	val-logloss:0.01236
rank                       0.3622
q_gap_next                 0.3229
q_best_ratio               0.0653
a_tset_gap_qmax            0.0342
num_tset                   0.0280
num_lq                     0.0256
n_squash_ratio_gap_qmax    0.0222
n_missing_q                0.0186
num_jac    

## 6. Test inference → `output/matching_results.tsv` + `output/candidate_pairs.tsv`

In [8]:
def write_lists(path, col, s1_ids, q_ids, si, qi, block_size=200_000):
    # one row per S1 (input order), comma-joined S2/S3 ids; strings built per block to save RAM
    o = np.lexsort((qi, si)); si, qi = si[o], qi[o]
    keep = np.r_[True, (si[1:] != si[:-1]) | (qi[1:] != qi[:-1])]
    si, qi = si[keep], qi[keep]
    bounds = np.searchsorted(si, np.arange(len(s1_ids) + 1))
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"source1_entity_id\t{col}\n")
        for a in range(0, len(s1_ids), block_size):
            b = min(a + block_size, len(s1_ids))
            ids = q_ids[qi[bounds[a]:bounds[b]]]
            loc = bounds[a:b + 1] - bounds[a]
            f.write("".join(f"{s1_ids[i]}\t{','.join(ids[loc[i - a]:loc[i - a + 1]])}\n" for i in range(a, b)))


s1, q = load_split("test")
if DEBUG:
    rng = np.random.default_rng(1)
    s1 = s1[rng.random(len(s1)) < 0.05].reset_index(drop=True)
    q = q[rng.random(len(q)) < 0.05].reset_index(drop=True)
apply_lexicon(q, LEX)
PAIRS_PATH = os.path.join(CACHE, "test_pairs.parquet")
if not os.path.exists(PAIRS_PATH):
    block(s1, q).to_parquet(PAIRS_PATH, index=False)
P = pd.read_parquet(PAIRS_PATH)
log(f"test pairs {len(P)} ({len(P) / len(q):.2f}/query), RAM {ram()}")

bst = xgb.Booster(); bst.load_model(MODEL_PATH)
bst.set_param({"device": "cuda" if GPU else "cpu", "nthread": N_JOBS})
PROB_PATH = os.path.join(CACHE, "test_prob.npy")
if os.path.exists(PROB_PATH):
    prob = np.load(PROB_PATH)
else:
    prob = np.zeros(len(P), dtype=np.float32)
    q_ctry, s_ctry = q.country.astype(str).to_numpy(), s1.country.astype(str).to_numpy()
    pair_ctry = q_ctry[P.qi.to_numpy()]
    for c in np.unique(pair_ctry):                      # one country at a time keeps RAM low
        rows = np.flatnonzero(pair_ctry == c)
        s_idx, q_idx = np.flatnonzero(s_ctry == c), np.flatnonzero(q_ctry == c)
        s_loc = np.full(len(s1), -1, np.int64); s_loc[s_idx] = np.arange(len(s_idx))
        q_loc = np.full(len(q), -1, np.int64); q_loc[q_idx] = np.arange(len(q_idx))
        Pc = P.iloc[rows].copy()
        Pc["qi"], Pc["si"] = q_loc[Pc.qi.to_numpy()], s_loc[Pc.si.to_numpy()]
        feat = Featurizer(s1.iloc[s_idx].reset_index(drop=True), q.iloc[q_idx].reset_index(drop=True))
        for a, b in chunk_bounds(Pc.qi.to_numpy(), 1_000_000):
            X = feat.transform(Pc.iloc[a:b])[REPORT["features"]].to_numpy(dtype=np.float32)
            prob[rows[a:b]] = bst.inplace_predict(X, iteration_range=(0, REPORT["best_iteration"] + 1))
        log(f"  scored {c}: {len(rows)} pairs, RAM {ram()}")
        del feat, Pc; gc.collect()
    np.save(PROB_PATH, prob)

THRESHOLD = REPORT["threshold"]           # change here to trade precision vs recall, then re-run this cell
pqi, psi = P.qi.to_numpy(), P.si.to_numpy()
sel = decode(pqi, prob, THRESHOLD)
s1id = s1.entity_id.astype(str).to_numpy()
qid = q.entity_id.astype(str).to_numpy()
write_lists(os.path.join(OUT, "matching_results.tsv"), "matched_entity_ids", s1id, qid, psi[sel], pqi[sel])
write_lists(os.path.join(OUT, "candidate_pairs.tsv"), "candidate_entity_ids", s1id, qid, psi, pqi)
n = np.bincount(psi[sel], minlength=len(s1))
ctry = s1.country.astype(str).to_numpy()
for c in np.unique(ctry):
    m = ctry == c
    log(f"  {c}: {m.sum()} S1, mean matches {n[m].mean():.2f}, empty rate {(n[m] == 0).mean():.3f}")
log(f"threshold {THRESHOLD}: wrote {OUT}/matching_results.tsv and candidate_pairs.tsv")

[  20.5 min]   block India: 809986 S1, 4717565 queries (4550048 unique docs)
[  37.3 min]   block India: done, 31100935 pairs, RAM 10.9 GB
[  37.6 min]   block France: 259452 S1, 1434993 queries (1315333 unique docs)
[  38.4 min]   block France: done, 13743661 pairs, RAM 9.7 GB
[  38.9 min]   block US: 663106 S1, 3817031 queries (3532511 unique docs)
[  44.0 min]   block US: done, 21865095 pairs, RAM 10.7 GB
[  44.5 min] test pairs 66709691 (6.69/query), RAM 9.4 GB


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [17:57:54] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[  48.6 min]   scored France: 13743661 pairs, RAM 11.7 GB
[  57.4 min]   scored India: 31100935 pairs, RAM 15.3 GB
[  62.6 min]   scored US: 21865095 pairs, RAM 13.6 GB
[  64.1 min]   France: 259452 S1, mean matches 3.51, empty rate 0.045
[  64.1 min]   India: 809986 S1, mean matches 3.50, empty rate 0.049
[  64.1 min]   US: 663106 S1, mean matches 3.53, empty rate 0.049
[  64.1 min] threshold 0.31: wrote /kaggle/working/output/matching_results.tsv and candidate_pairs.tsv


## 7. Validate the submission files

In [9]:
m = pd.read_csv(os.path.join(OUT, "matching_results.tsv"), sep="\t", dtype=str, keep_default_na=False)
assert list(m.columns) == ["source1_entity_id", "matched_entity_ids"]
assert m.source1_entity_id.is_unique and set(m.source1_entity_id) == set(s1id), "S1 coverage mismatch"
lists = m.matched_entity_ids[m.matched_entity_ids != ""].str.split(",")
assert lists.map(lambda l: len(l) == len(set(l))).all(), "duplicate ids in a list"
assert lists.explode().str.match(r"^S[23]-").all(), "non S2/S3 id"
print(f"basic checks OK: {len(m)} rows, {(m.matched_entity_ids == '').sum()} empty")

val = glob.glob("/kaggle/input/**/validate_submission.py", recursive=True) + glob.glob("utils/validate_submission.py")
if val and not DEBUG:
    subprocess.run([sys.executable, val[0], "--matching", os.path.join(OUT, "matching_results.tsv"),
                    "--candidate", os.path.join(OUT, "candidate_pairs.tsv"),
                    "--test-dir", os.path.join(DATA, "test")], check=False)
else:
    print("official validator not found (add utils/validate_submission.py to the dataset to run it)")
print("Download /kaggle/working/output/matching_results.tsv and upload it to the portal.")

basic checks OK: 1732544 rows, 83719 empty
ML Challenge 2026 — submission validator
  test dir: /kaggle/input/datasets/gouravanirudh/amazon-ml-challenge/student_resource/dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (83719 empty, 1648825 non-empty).
  candidate_pairs.tsv: 1732544 rows (124 empty, 1732420 non-empty).

PASS — no blocking issues found. Safe to submit.
Download /kaggle/working/output/matching_results.tsv and upload it to the portal.
